# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhruvaiyer2-netizen/FlyRankMLProject/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The source for the data from the table is page-per-day, but for my decision unit, I intend to aggregate it into as page-per-month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

impressions, clicks, ctr, avg_position/position, position_tier, sessions, engagement_rate, scroll_rate, word_count, content type, main_intent, age/freshness are all in the features bucket.
ga4_data_available, report_date, client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, gsc_data_start, ga4_data_start are all in the context bucket.
tier-relative ctr gap (will be derived by me) belongs in the label/proxy bucket.
health_score, priority_score, action_type, needs_ctr_fix, is_quick_win, refresh_tier are generated by flyrank, so feeding this to a model would just teach it to copy the product instead of actually learning.
raw query, urls, title domain text, client text are all scrambled before release so they do not bias or influence the training of the model, not to mention they are confidential pieces of data.
AI Session fields (sessions_ai) is too sparse to support the chosen lane.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The trap: Using only features independent of the label (avg_position, impressions_90d, word_count), a random forest classifier predicting "is this page flagged under the rule" scored 0.572 precision — a modest, believable result above the 0.500 base rate. Adding ctr back in as a feature — the exact column the label (ctr < tier_median_ctr) was built from — pushed precision to 1.000, a jump of +0.428. This isn't the model learning a real pattern; it's reading the label's own definition back out of the input. The honest number, 0.572, is the one that reflects genuine predictive signal and is the only one that should be reported or compared against the rule-based baseline.

In [9]:
# ============================================================
# SETUP & AUTH
# ============================================================
!pip install -q duckdb huggingface_hub

from huggingface_hub import login
login()  # paste your HF token when prompted

import duckdb
import pandas as pd
from huggingface_hub import HfFileSystem
duckdb.register_filesystem(HfFileSystem())

# ============================================================
# STEP 1: LOAD MARCH 2026 FACT DATA
# ============================================================
duckdb.sql("""
    CREATE OR REPLACE TABLE march_facts AS
    SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
    WHERE month = '2026-03'
""")

duckdb.sql("SHOW TABLES").show()
duckdb.sql("SELECT COUNT(*) AS n_rows FROM march_facts").show()

# ============================================================
# STEP 2: VERIFY THE THREE CONTRACT FACTS
# ============================================================
print("=== 1. GRAIN CHECK ===")
duckdb.sql("""
    SELECT
      COUNT(*) AS total_rows,
      COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_keys
    FROM march_facts
""").show()

print("=== 2. ROW COUNT + DATE SPAN ===")
duckdb.sql("""
    SELECT
      COUNT(*) AS n_rows,
      MIN(report_date) AS earliest,
      MAX(report_date) AS latest,
      COUNT(DISTINCT client_hash_id) AS n_clients,
      COUNT(DISTINCT content_hash_id) AS n_pages
    FROM march_facts
""").show()

print("=== 3. AVAILABILITY (IS TRUE) ===")
duckdb.sql("""
    SELECT
      COUNT(*) AS total_march_rows,
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM march_facts
""").show()

# ============================================================
# STEP 3: AGGREGATE DAILY FACTS -> ONE ROW PER PAGE
# (impression-weighted CTR and position — ratio-of-sums, not average-of-averages)
# ============================================================
pages_df = duckdb.sql("""
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(gsc_impressions) AS impressions_90d,
      SUM(gsc_clicks) AS clicks_90d,
      SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
      100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM march_facts
    GROUP BY content_hash_id, client_hash_id
""").df()

print(f"Aggregated to {len(pages_df)} page-level rows")

# ============================================================
# STEP 4: ADD POSITION TIER
# ============================================================
pages_df["position_tier"] = pd.cut(
    pages_df["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)

# ============================================================
# STEP 5: LOAD CONTENT METADATA, JOIN, DROP DELETED PAGES
# ============================================================
dim_content_df = duckdb.sql("""
    SELECT content_hash_id, word_count, main_intent, content_type, is_published, is_deleted
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").df()

print(f"dim_content rows: {len(dim_content_df)}")
print(f"Unique content_hash_id in dim_content: {dim_content_df['content_hash_id'].nunique()}")

feature_frame = pages_df.merge(dim_content_df, on="content_hash_id", how="left")

# Drop pages that no longer exist — can't review/action a deleted page
feature_frame = feature_frame[feature_frame["is_deleted"] != True].copy()

print(f"Rows after dropping deleted pages: {len(feature_frame)}")
print(f"Rows still missing word_count: {feature_frame['word_count'].isna().sum()}")

# ============================================================
# STEP 6: FINAL FIVE-FEATURE FRAME
# ============================================================
five_features = feature_frame[[
    "content_hash_id", "avg_position", "position_tier",
    "impressions_90d", "word_count", "main_intent"
]]

print(f"Final feature frame: {len(five_features)} rows")
five_features.head(10)

duckdb.sql("""
    SELECT
      is_published, is_deleted,
      COUNT(*) AS n_rows,
      COUNT(word_count) AS n_with_word_count,
      COUNT(*) - COUNT(word_count) AS n_missing_word_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    GROUP BY is_published, is_deleted
""").show()

published_live = duckdb.sql("""
    SELECT
      content_type,
      COUNT(*) AS n_rows,
      COUNT(word_count) AS n_with_word_count,
      COUNT(*) - COUNT(word_count) AS n_missing
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    WHERE is_published = true AND is_deleted = false
    GROUP BY content_type
    ORDER BY n_missing DESC
""").show()

duckdb.sql("""
    SELECT
      EXTRACT(YEAR FROM content_created_date) AS created_year,
      COUNT(*) AS n_rows,
      COUNT(word_count) AS n_with_word_count,
      COUNT(*) - COUNT(word_count) AS n_missing
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    WHERE is_published = true AND is_deleted = false
    GROUP BY created_year
    ORDER BY created_year
""").show()
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

MIN_IMPRESSIONS = 500

model_data = feature_frame.dropna(subset=["avg_position", "impressions_90d", "ctr"]).copy()
model_data = model_data[model_data["impressions_90d"] >= MIN_IMPRESSIONS].copy()  # <-- the missing piece

model_data["tier"] = pd.cut(
    model_data["avg_position"], bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"]
)
tier_median_ctr = model_data.groupby("tier", observed=True)["ctr"].transform("median")
model_data["is_flagged"] = (model_data["ctr"] < tier_median_ctr).astype(int)

print(f"Rows after volume filter: {len(model_data)}")
print(f"Base rate: {model_data['is_flagged'].mean():.3f} of pages flagged")

# ---- HONEST features only ----
honest_features = ["avg_position", "impressions_90d"]
model_data["word_count_filled"] = model_data["word_count"].fillna(-1)
honest_features.append("word_count_filled")

X_honest = model_data[honest_features]
y = model_data["is_flagged"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
clf_honest.fit(X_train, y_train)
honest_precision = precision_score(y_test, clf_honest.predict(X_test), zero_division=0)
print(f"\nHONEST score (no label-derived features): precision = {honest_precision:.3f}")

# ---- THE TRAP: add ctr itself ----
leaky_features = honest_features + ["ctr"]
X_leaky = model_data[leaky_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
clf_leaky = RandomForestClassifier(n_estimators=100, random_state=42)
clf_leaky.fit(X_train_l, y_train_l)
leaky_precision = precision_score(y_test_l, clf_leaky.predict(X_test_l), zero_division=0)
print(f"LEAKY score (ctr included):            precision = {leaky_precision:.3f}")

print(f"\nJump from leakage: {leaky_precision - honest_precision:+.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ march_facts │
└─────────────┘

┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 9841378 │
└─────────┘

=== 1. GRAIN CHECK ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ total_rows │ unique_keys │
│   int64    │    int64    │
├────────────┼─────────────┤
│    9841378 │     9841378 │
└────────────┴─────────────┘

=== 2. ROW COUNT + DATE SPAN ===
┌─────────┬────────────┬────────────┬───────────┬─────────┐
│ n_rows  │  earliest  │   latest   │ n_clients │ n_pages │
│  int64  │    date    │    date    │   int64   │  int64  │
├─────────┼────────────┼────────────┼───────────┼─────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │        55 │  331437 │
└─────────┴────────────┴────────────┴───────────┴─────────┘

=== 3. AVAILABILITY (IS TRUE) ===
┌──────────────────┬────────────────────┐
│ total_march_rows │ ga4_available_rows │
│      int64       │       int64        │
├──────────────────┼────────────────────┤
│          9841378 │             413966 │
└──────────────────┴────────────────────┘

Aggregated to 331437 page-level rows
dim_content rows: 519606
Unique content_hash_id in dim_content: 519606
Rows after dropping deleted pages: 3

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me a single page's typical behaviour across clients, because history length is unbalanced across the 55 clients in this slice, nor can it tell me about enagement or session behaviour for all the rows with ga4_data_available = False. It also cannot tell me any future prediction claim if the feature and label window share days - that would leak the answer into the input rather than test whether or not the model can find it independently. All three points mentioned above are structural limits of the data rather than somethingt that can be fixed by better models or more rows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.